In [10]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Changed import

documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embed_model.encode(documents)

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

query = "What is RAG in AI?"
query_embedding = embed_model.encode([query])

D, I = index.search(np.array(query_embedding), k=2)

retrieved_chunks = [documents[i] for i in I[0]]

context = " ".join(retrieved_chunks)

prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

# Load tokenizer and model directly (instead of pipeline)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# Prepare input for the model
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Generate text
generated_ids = model.generate(input_ids, max_new_tokens=60, do_sample=False)
answer_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print("Retrieved Context:", retrieved_chunks)
print("Answer:", answer_text)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Retrieved Context: ['Python is a popular high-level programming language used in AI development.', 'Retrieval-Augmented Generation combines document retrieval with text generation.']
Answer: combines document retrieval with text generation
